# Check if features producing noise accross development set

- Conclusion: No common features across lower ends of outcome/model pairs (nothing to remove)

In [ ]:
import os, sys

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH, DEVICE, SEED

print(f"Path: {BASE_PATH}")

In [ ]:
OUTCOME_LIST = [
    "ssi",
    "serious",
    "any",
    "pneumo",
    "bleed",
    "reoper",
]
MODEL_ABRV_LIST = [
    # nn added after
    "lr",
    "lgbm",
    "svc",
    "xgb",
    "stack",
]

IMP_DIR = BASE_PATH / "data" / "processed_reduced"
MODEL_DIR = BASE_PATH / "models" / "trained"

In [ ]:
from src.data_utils import get_data, get_models
from sklearn.inspection import permutation_importance
from src.nn_models import load_nn_clf
import pandas as pd
import matplotlib.pyplot as plt

for outcome in OUTCOME_LIST:
    print(outcome)
    data_dict = get_data(outcome_folder=f"outcome_{outcome}", file_dir=IMP_DIR)
    X_val, y_val = data_dict["X_val"], data_dict["y_val"].values.ravel()
    model_dir = MODEL_DIR / outcome
    model_dict = get_models(MODEL_ABRV_LIST, model_dir)
    ## Load NN
    nn_in_dim = X_val.shape[1]
    nn_dir = MODEL_DIR / outcome / "nn.pt"
    model_dict["nn"] = load_nn_clf(data_path=nn_dir, in_dim=nn_in_dim, device=DEVICE)
    for model_name, model in model_dict.items():
        print(f"\t {model_name}")
        result = permutation_importance(
            estimator=model,
            X=X_val,
            y=y_val,
            scoring="average_precision",
            n_repeats=50,
            random_state=SEED,
        )
        mean_imp = result["importances_mean"]
        stdev_imp = result["importances_std"]
        feat_names = X_val.columns.to_list()
        perm_df = (
            pd.DataFrame(
                {"Feature": feat_names, "Mean Imp": mean_imp, "Stdev Imp": stdev_imp}
            )
            .sort_values(by="Mean Imp", ascending=True)
            .reset_index(drop=True)
        )
        plt.figure()
        plt.figure(figsize=(6, 14))
        plt.barh(
            perm_df["Feature"],
            perm_df["Mean Imp"],
            yerr=perm_df["Stdev Imp"],
            capsize=4,
        )

        plt.xlabel("Mean Importance")
        plt.ylabel("Feature")
        plt.title(f"Outcome: {outcome}\nModel: {model_name}")
        plt.tight_layout()
        plt.show()